[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/qgis_workflow_processing.ipynb)

# QGIS Workflow and Batch Extraction Concepts

This notebook demonstrates the Python workflow logic used by the QGIS Workflow Builder and the long-form output structure used by the batch spectral extraction Processing tool.

In [ ]:
# %pip install hypercoast

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import hypercoast

## Create a workflow-ready cube

In [ ]:
dataset = xr.Dataset(
    {
        "reflectance": (
            ("wavelength", "y", "x"),
            np.array(
                [
                    [[0.05, 0.08, 0.12], [0.04, 0.07, 0.10]],
                    [[0.12, 0.14, 0.18], [0.10, 0.13, 0.16]],
                    [[0.03, 0.04, 0.06], [0.02, 0.03, 0.05]],
                    [[0.01, 0.02, 0.04], [0.01, 0.02, 0.03]],
                ],
                dtype="float32",
            ),
        )
    },
    coords={"wavelength": [412.0, 560.0, 665.0, 860.0], "y": [0, 1], "x": [0, 1, 2]},
)
dataset

## Run the same presets exposed in QGIS

The QGIS Workflow Builder uses the same `hypercoast.apply_workflow()` presets exposed to Python and the CLI.

In [ ]:
hypercoast.list_workflows()

In [ ]:
ndwi = hypercoast.apply_workflow(dataset, "ndwi", variable="reflectance")
turbidity = hypercoast.apply_workflow(dataset, "turbidity", variable="reflectance")
cdom = hypercoast.apply_workflow(dataset, "cdom", variable="reflectance")

ndwi.name, turbidity.name, cdom.name

In [ ]:
ndwi

The QGIS dock writes this 2D workflow output to a GeoTIFF and adds it back to the current project.

## Override workflow wavelengths

The Workflow Builder includes optional wavelength override fields. In Python, pass a `wavelengths` dictionary with `a` and `b` values.

In [ ]:
custom_ratio = hypercoast.apply_workflow(
    dataset,
    "turbidity",
    variable="reflectance",
    wavelengths={"a": 665.0, "b": 860.0},
)
custom_ratio

## Batch extraction output structure

The QGIS Processing batch extractor writes one row per feature and wavelength. The table below shows the expected long-form shape.

In [ ]:
points = pd.DataFrame(
    {
        "feature_id": [0, 1],
        "x": [-90.6, -90.4],
        "y": [28.7, 28.8],
        "crs": ["EPSG:4326", "EPSG:4326"],
    }
)

rows = []
for _, point in points.iterrows():
    # In QGIS this vector is returned by HyperspectralDataset.extract_spectral_signature().
    values = np.array([0.05, 0.12, 0.03, 0.01]) + point.feature_id * 0.01
    for wavelength, value in zip(dataset.wavelength.values, values):
        rows.append(
            {
                "feature_id": point.feature_id,
                "x": point.x,
                "y": point.y,
                "crs": point.crs,
                "wavelength": wavelength,
                "value": value,
                "layer": "synthetic_cube",
                "variable": "reflectance",
            }
        )

spectra = pd.DataFrame(rows)
spectra

In QGIS, use **Processing** > **Toolbox** > **HyperCoast** > **Batch Extract Spectra** with a point CSV and an output CSV path.